In [2]:
import pandas as pd
import os

In [3]:
df = pd.read_csv("dataset/raw/adult_drifted_invalid.csv", index_col=False)

In [4]:
for col in df.columns:
    print(f"=============[{col}]===================")
    print(df[col].value_counts(dropna=False).sort_index())
    print(f"Null: {df[col].isnull().sum()}")
    print()
    print()

=============[age]===================
-49     27
-48     18
-47     20
-46     27
-45     14
        ..
 96     26
 97     14
 98     21
 99     22
 100    20
Name: age, Length: 128, dtype: int64
Null: 0


=============[workclass]===================
?                    5629
Federal-gov          2907
Local-gov            6278
Never-worked           14
Private             69832
Self-emp-inc         3403
Self-emp-not-inc     7847
State-gov            4042
Without-pay            48
Name: workclass, dtype: int64
Null: 0


=============[fnlwgt]===================
12285      6
13769      2
14878      6
18827      4
19214      1
          ..
1226583    2
1268339    4
1366120    3
1455435    1
1484705    4
Name: fnlwgt, Length: 20943, dtype: int64
Null: 0


=============[education]===================
10th             2817
11th             3646
12th             1363
1st-4th           510
5th-6th          1013
7th-8th          1931
9th              1588
Assoc-acdm       3293
Assoc-voc        431

In [5]:
string_cols = df.select_dtypes(include='object').columns
df[string_cols] = df[string_cols].fillna('unknown')

In [6]:
numeric_cols = df.select_dtypes(include='number').columns
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(0)

In [8]:
#os.makedirs('dataset/clean')
df.to_csv('dataset/clean/adult2.csv', index=False)

In [9]:
import tensorflow as tf
print('TF: {}'.format(tf.__version__))
import numpy as np
import tensorflow_data_validation as tfdv
print('TFDV version:', tfdv.version.__version__)
from tensorflow_metadata.proto.v0 import schema_pb2

2025-06-26 22:24:17.836030: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-26 22:24:17.941813: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-26 22:24:18.060240: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-26 22:24:18.175241: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-26 22:24:18.175953: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-26 22:24:18.352230: I tensorflow/core/platform/cpu_feature_guard.cc:

TF: 2.16.2
TFDV version: 1.16.1


In [10]:
raw_df = pd.read_csv('dataset/clean/adult2.csv')
print(raw_df.columns)
print(f"Loaded {len(raw_df):,} rows.")

Index(['age', 'workclass', 'fnlwgt', 'education', 'education.num',
       'marital.status', 'occupation', 'relationship', 'race', 'sex',
       'capital.gain', 'capital.loss', 'hours.per.week', 'native.country',
       'income'],
      dtype='object')
Loaded 100,000 rows.


In [11]:
# Select numeric columns
numeric_df = raw_df.select_dtypes(include=[np.number])

summary = pd.DataFrame({
    'min': numeric_df.min(),
    'max': numeric_df.max(),
    'mean': numeric_df.mean(),
    'median': numeric_df.median(),
    #'mode': numeric_df.mode()
})
print(summary)

                  min      max          mean    median
age               -49      100      45.31271      44.0
fnlwgt          12285  1484705  189554.79603  177955.0
education.num       1       16      10.07973      10.0
capital.gain        0    99999    1081.58272       0.0
capital.loss        0     4356      86.16632       0.0
hours.per.week    -39      104      43.27100      44.0


In [12]:
stats = tfdv.generate_statistics_from_csv(data_location='dataset/clean/adult.csv')
schema = tfdv.infer_schema(stats)

Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


In [13]:
#schema = schema_pb2.Schema()

age_feature = schema.feature.add()
age_feature.name = 'age'
age_feature.type = schema_pb2.FeatureType.INT
age_feature.int_domain.min = 17
age_feature.int_domain.max = 90
age_feature.drift_comparator.jensen_shannon_divergence.threshold = 0.1  # 10% proportion change allowed

education_num_feature = schema.feature.add()
education_num_feature.name = 'education.num'
education_num_feature.type = schema_pb2.FeatureType.INT
education_num_feature.int_domain.min = 1
education_num_feature.drift_comparator.jensen_shannon_divergence.threshold = 0.1  # 10% proportion change allowed

capital_gain_feature = schema.feature.add()
capital_gain_feature.name = 'capital.gain'
capital_gain_feature.type = schema_pb2.FeatureType.INT
capital_gain_feature.int_domain.min = 0
capital_gain_feature.drift_comparator.jensen_shannon_divergence.threshold = 0.1  # 10% proportion change allowed

capital_loss_feature = schema.feature.add()
capital_loss_feature.name = 'capital.loss'
capital_loss_feature.type = schema_pb2.FeatureType.INT
capital_loss_feature.int_domain.min = 0
capital_loss_feature.drift_comparator.jensen_shannon_divergence.threshold = 0.1  # 10% proportion change allowed

hrs_per_week_feature = schema.feature.add()
hrs_per_week_feature.name = 'hours.per.week'
hrs_per_week_feature.type = schema_pb2.FeatureType.INT
hrs_per_week_feature.int_domain.min = 1
hrs_per_week_feature.int_domain.max = 99
hrs_per_week_feature.drift_comparator.jensen_shannon_divergence.threshold = 0.1  # 10% proportion change allowed

fnlwgt = schema.feature.add()
fnlwgt.name = 'fnlwgt'
fnlwgt.type = schema_pb2.FeatureType.INT

# education = schema.feature.add()
# education.name = 'education'
# education.type = schema_pb2.FeatureType.BYTES

# sex = schema.feature.add()
# sex.name = 'sex'
# sex.type = schema_pb2.FeatureType.BYTES

# nc = schema.feature.add()
# nc.name = 'native.country'
# nc.type = schema_pb2.FeatureType.BYTES

# relationship = schema.feature.add()
# relationship.name = 'relationship'
# relationship.type = schema_pb2.FeatureType.BYTES

# occupation = schema.feature.add()
# occupation.name = 'occupation'
# occupation.type = schema_pb2.FeatureType.BYTES

# marital_status = schema.feature.add()
# marital_status.name = 'marital.status'
# marital_status.type = schema_pb2.FeatureType.BYTES

# race = schema.feature.add()
# race.name = 'race'
# race.type = schema_pb2.FeatureType.BYTES

# workclass = schema.feature.add()
# workclass.name = 'workclass'
# workclass.type = schema_pb2.FeatureType.BYTES

# income = schema.feature.add()
# income.name = 'income'
# income.type = schema_pb2.FeatureType.BYTES

In [14]:
tfdv.write_schema_text(schema, 'schema.pbtxt')

In [15]:
schema

feature {
  name: "age"
  presence {
    min_fraction: 1
    min_count: 1
  }
  shape {
    dim {
      size: 1
    }
  }
  type: INT
}
feature {
  name: "workclass"
  presence {
    min_fraction: 1
    min_count: 1
  }
  shape {
    dim {
      size: 1
    }
  }
  type: BYTES
  domain: "workclass"
}
feature {
  name: "fnlwgt"
  presence {
    min_fraction: 1
    min_count: 1
  }
  shape {
    dim {
      size: 1
    }
  }
  type: INT
}
feature {
  name: "education"
  presence {
    min_fraction: 1
    min_count: 1
  }
  shape {
    dim {
      size: 1
    }
  }
  type: BYTES
  domain: "education"
}
feature {
  name: "education.num"
  presence {
    min_fraction: 1
    min_count: 1
  }
  shape {
    dim {
      size: 1
    }
  }
  type: INT
}
feature {
  name: "marital.status"
  presence {
    min_fraction: 1
    min_count: 1
  }
  shape {
    dim {
      size: 1
    }
  }
  type: BYTES
  domain: "\'marital.status\'"
}
feature {
  name: "occupation"
  presence {
    min_fraction: 1
   